<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/05_augment_sentiment_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Augmenting Financial Sentiment Data with LLM-as-a-Judge

In this notebook, we implement a robust **Data Augmentation** pipeline for financial sentiment analysis. Beyond simple paraphrasing, we use a powerful teacher model to anonymize data and diversify numerical figures, followed by an **LLM-as-a-Judge** phase to ensure the quality of our synthetic examples.

### Why Augment with a Teacher Model?
- **Robustness**: Replacing specific PII (names, locations) and changing figures prevents the model from "memorizing" specific entities and forces it to learn the underlying financial sentiment patterns.
- **Data Scarcity**: Financial datasets are often small due to the high cost of expert annotation. Augmentation helps us "hallucinate" valid training examples.
- **Diversity**: By rephrasing headlines, we expose the student model to various linguistic styles and professional jargon.

### The LLM-as-a-Judge Concept
To maintain high data quality, we don't just trust the augmentation blindly. We use the teacher model again (or a different one) as a **Judge**. The Judge evaluates each augmented pair based on specific criteria like sentiment preservation and PII removal, assigning a score that we can use to filter out low-quality synthetic data.

### Technical Approach
We utilize **Qwen 2.5 7B** with **4-bit NF4 quantization**. This allows us to run a high-performance instructor model on accessible hardware (like a T4 or L4 GPU) for both generation and evaluation.

In [1]:
%%capture
import os

if "COLAB_" in "".join(os.environ.keys()):
    !pip install -U transformers trl accelerate bitsandbytes

In [2]:
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
import warnings
import json
import re

warnings.filterwarnings("ignore")

## 1. Configuration

We set up our model ID and quantization parameters. We use a batch size of 8 to balance throughput with the increased memory pressure of handling multiple generated variations.

In [3]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DATASET_ID = "lmassaron/FinancialPhraseBank"
OUTPUT_PATH = "FinancialPhraseBank_augmented_judged"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 512

QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}

## 2. Prompt Engineering

We define two distinct system prompts:
1. **Augmenter**: Focused on creative rewriting, PII removal, and figure diversification.
2. **Judge**: Focused on objective evaluation against strict quality criteria.

In [4]:
AUGMENTER_SYSTEM_PROMPT = (
    "You are an expert financial data scientist specializing in data augmentation. "
    "Your task is to take a financial news headline and create 2 distinct variations of it. "
    "For each variation:\n"
    "1. Replace PII (names, specific locations, small companies) with realistic but fake or generic equivalents.\n"
    "2. Replace numerical figures with different but plausible values that preserve the original market implication.\n"
    "3. Rephrase the sentence completely while strictly maintaining the original sentiment.\n"
    'Format your output as a JSON list of strings: ["Variation 1", "Variation 2"]'
)

JUDGE_SYSTEM_PROMPT = (
    "You are a quality control judge for financial data augmentation. "
    "Compare the 'Original' headline with its 'Augmented' version. "
    "Evaluate based on:\n"
    "1. PII Removal: Names/locations/companies are replaced?\n"
    "2. Figure Diversification: Numbers are different but same implication?\n"
    "3. Rephrasing: Sentence structure is different?\n"
    "4. Sentiment Preservation: Original sentiment is strictly maintained?\n\n"
    "Provide a Score (0-10) and a brief justification. "
    "End your response with 'Final Score: X/10' where X is the score."
)


def build_augment_prompt(sentence: str, sentiment: str) -> str:
    return (
        f'Original Headline: "{sentence}"\n'
        f"Sentiment: {sentiment}\n\n"
        f"Generate 2 augmented variations in JSON format."
    )


def build_judge_prompt(original: str, augmented: str, sentiment: str) -> str:
    return (
        f"Original: {original}\n"
        f"Augmented: {augmented}\n"
        f"Expected Sentiment: {sentiment}\n\n"
        "Evaluate the quality of this augmentation."
    )

## 3. Loading the Model

We load Qwen 2.5 using `device_map="auto"` to utilize available VRAM efficiently.

In [5]:
def load_model(model_id: str):
    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
    )
    model.eval()
    return tokenizer, model


tokenizer, model = load_model(MODEL_ID)

Loading Qwen/Qwen2.5-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 4. Augmentation & Judging Pipeline

This section handles the core logic. Note how we use `batch_decode` and token slicing to extract only the model's new response.

In [6]:
def generate_batch(tokenizer, model, chats):
    inputs = tokenizer(
        chats, return_tensors="pt", padding=True, truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)


def parse_variations(raw_output):
    try:
        start = raw_output.find("[")
        end = raw_output.rfind("]") + 1
        if start != -1 and end != -1:
            return json.loads(raw_output[start:end])
    except:
        pass
    return []


def parse_judge_score(judgment):
    match = re.search(r"Final Score: (\d+)/10", judgment)
    if match:
        return int(match.group(1))
    return 5  # Default neutral score if parsing fails

## 5. Dataset Processing

We iterate through the dataset, generate variations, and then immediately judge them. This allows us to keep the "Augmented" and "Judgment" metadata together.

In [7]:
def process_dataset(tokenizer, model, dataset, limit=None):
    results = []
    sentences = dataset["sentence"]
    labels = dataset["label"]
    if limit:
        sentences, labels = sentences[:limit], labels[:limit]

    for start in tqdm(
        range(0, len(sentences), BATCH_SIZE), desc="Augmenting & Judging"
    ):
        # 1. Generate Augmentations
        batch_sentences = sentences[start : start + BATCH_SIZE]
        batch_labels = labels[start : start + BATCH_SIZE]

        aug_chats = [
            tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": AUGMENTER_SYSTEM_PROMPT},
                    {"role": "user", "content": build_augment_prompt(s, LABEL_MAP[l])},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for s, l in zip(batch_sentences, batch_labels)
        ]

        aug_raw = generate_batch(tokenizer, model, aug_chats)

        for i, (orig_s, label, raw) in enumerate(
            zip(batch_sentences, batch_labels, aug_raw)
        ):
            variations = parse_variations(raw)
            sentiment = LABEL_MAP[label]

            # Add original
            results.append(
                {
                    "sentence": orig_s,
                    "label": label,
                    "is_augmented": False,
                    "judge_score": 10,
                    "judge_comment": "Original data",
                }
            )

            if not variations:
                continue

            # 2. Judge each variation
            judge_chats = [
                tokenizer.apply_chat_template(
                    [
                        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                        {
                            "role": "user",
                            "content": build_judge_prompt(orig_s, v, sentiment),
                        },
                    ],
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for v in variations
            ]

            judgments = generate_batch(tokenizer, model, judge_chats)

            for v, j in zip(variations, judgments):
                results.append(
                    {
                        "sentence": v,
                        "label": label,
                        "is_augmented": True,
                        "judge_score": parse_judge_score(j),
                        "judge_comment": j,
                    }
                )
    return results

## 6. Execution

We process the dataset splits. For demonstration, you might want to `limit` the number of examples if running on a slow connection or restricted environment.

In [8]:
raw = load_dataset(DATASET_ID)
output_splits = {}

for split_name, data in raw.items():
    print(f"\n── Processing split: {split_name} ──")
    # Limiting to 50 for quick demonstration, remove limit for full dataset
    results = process_dataset(tokenizer, model, data, limit=3)
    output_splits[split_name] = Dataset.from_list(results)

output_ds = DatasetDict(output_splits)
output_ds.save_to_disk(OUTPUT_PATH)
print(f"\nAugmented and judged dataset saved to: {OUTPUT_PATH}")


── Processing split: train ──


Augmenting & Judging:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



── Processing split: validation ──


Augmenting & Judging:   0%|          | 0/1 [00:00<?, ?it/s]

...

Augmenting & Judging:   0%|          | 0/1 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/9 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9 [00:00<?, ? examples/s]


Augmented and judged dataset saved to: FinancialPhraseBank_augmented_judged


## 7. Quality Analysis

Finally, we can filter our dataset to keep only the highest quality augmentations (e.g., score >= 8). This ensures our student model learns from reliable examples.

In [9]:

df = output_ds["train"].to_pandas()
augmented_df = df[df["is_augmented"] == True]

print(f"Total augmented examples: {len(augmented_df)}")
print(f"Average Judge Score: {augmented_df['judge_score'].mean():.2f}/10")

# Show a high-quality example
best = augmented_df.sort_values("judge_score", ascending=False).iloc[0]
print("\n--- Top Quality Augmentation ---")
print(f"Sentence: {best['sentence']}")
print(f"Score   : {best['judge_score']}/10")
print(f"Comment : {best['judge_comment']}")

Total augmented examples: 6
Average Judge Score: 6.67/10

--- Top Quality Augmentation ---
Sentence: FictoBank plans to expand its presence in WestG Post locations, opening up 90 branches to offer financial services to the community.
Score   : 8/10
Comment : 1. PII Removal: The names "Post Bank" and "Fidelity Bank" are replaced with "Post" and "FictoBank", and "Ghana Post" is replaced with "WestG Post". This indicates that personal information has been removed or altered.
2. Figure Diversification: The original mentions 75 offices while the augmented version states 90 branches. Both numbers represent an expansion, maintaining the implication of growth.
3. Rephrasing: The sentence structure is quite different. The original uses "Known as" and directly states the concept, while the augmented version uses "plans to expand" and rephrases the location and service provision.
4. Sentiment Preservation: The original headline suggests a positive development (expansion and providing services), w

In [20]:
output_ds["train"]["judge_score"]

Column([10, 7, 5, 10, 6, ...])